In [18]:
from scipy.interpolate import BSpline

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from dataclasses import dataclass, field
from typing import List

from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import pickle
import os
import pandas as pd
import re
import joblib

In [19]:
# r for 22 sections
geometry_baseline_r = [0.0201254, 0.0252054, 0.0315579, 0.0379079, 0.0442579, 0.0506079, 0.0569579,
 0.0633079, 0.0696579, 0.0760079, 0.0823586, 0.0887148, 0.0950769, 0.1014424,
 0.107804,  0.1141792, 0.1173761, 0.1205864, 0.1238169, 0.1255,    0.12625,
 0.127]

# generate_bspline_basis_custom
### param : control_points(size: 1 * 8)
### return : matrices（3 * 22）--（r/R, chord, twist）


In [20]:
def generate_section(n_control_points):
    chord_points = n_control_points[0:4]
    twist_points = n_control_points[4:]
    degree = 3
    knots = np.concatenate(([0] * degree, [0.3984874, 0.89904882], [1] * degree))
    twist_knots = np.concatenate(([0] * degree, [0.2, 0.89904882], [1] * degree))

    chord_bspline = BSpline(knots, chord_points, degree)
    twist_bspline = BSpline(twist_knots, twist_points, degree)
    x_norm = [x / 0.127 for x in geometry_baseline_r]
    chord_spline = chord_bspline(x_norm)
    twist_spline = twist_bspline(x_norm)

    propeller_geometry = np.array([x_norm, chord_spline, twist_spline]).T
    return propeller_geometry


# Propeller Modeling Net

## congig for model

In [ ]:
class PropellerPredictor(nn.Module):
    """
    螺旋桨气动力预测网络
    输入: [RPM, WIND, ANGLE]  (3维)
    输出: [Fx, Fy, Fz, Torque] (4维)
    """
    def __init__(self, input_dim=3, hidden_dims=[64, 128, 128, 64], output_dim=4):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.BatchNorm1d(dims[i + 1]))
            layers.append(nn.ReLU())
            if i >= 2:
                layers.append(nn.Dropout(0.1))
        layers.append(nn.Linear(dims[-1], output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# traing

## create a tensorboard for recording the training process

In [22]:
log_dir = f'runs/propeller_predictor_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
writer = SummaryWriter(log_dir=log_dir)


## deal_data for trainging LFM model

In [23]:
# data_path = './data_for_train'
# data_files = os.listdir(data_path)
# all_data = {}
# for file in data_files:
#     if file.endswith('.pkl') and 'data' in str(file):
#         with open(os.path.join(data_path, file), 'rb') as f:
#             data = pickle.load(f)
#             all_data.update(data)

### data_structure
- keys:geometry_num
  - values: 
    - keys: RMP****_Wind**_Angle**
      - values: Time,Thrust,Power,Torch,Trust_y,Thrust_z -- dataframes

In [24]:
# columns_runningconditon = ['geometry_num', 'RPM', 'WIND', 'ANGLE',]
# chord_columns = [f'chord_{i}' for i in range(22)]
# twist_columns = [f'twist_{i}' for i in range(22)]
# output_columns = ['Power', 'Fx', 'Fy', 'Fz', 'Torque']
# columns = columns_runningconditon + chord_columns + twist_columns + output_columns
# data_total = pd.DataFrame(columns=columns) # create an empty dataframe to store all data

# # iteration all_data
# for geo_idx, (key, value) in enumerate(all_data.items()):
#     chord = None
#     twist = None
#     data_list = []

#     for key_, value_ in value.items():
#         # store the geometry
#         if 'geometry' in key_:
#             chord = value_[:, 1]
#             twist = value_[:, 2]
        
#         elif 'RPM' in key_:
#             # store the running condition
#             numbers = re.findall(r'\d+(?:\.\d+)?', key_) 
#             RPM, WIND, ANGLE = map(float, numbers)
#             data_one = pd.DataFrame(columns=columns)
#             data_one.at[0, 'geometry_num'] = geo_idx
#             data_one.at[0, 'RPM'] = RPM
#             data_one.at[0, 'WIND'] = WIND
#             data_one.at[0, 'ANGLE'] = ANGLE
#             # store the output data
#             data_one.at[0, 'Fx'] = -value_['Thrust'].mean() # (because of the direction of thrust)
#             data_one.at[0, 'Fy'] = value_['Trust_y'].mean()
#             data_one.at[0, 'Fz'] = value_['Trust_z'].mean()
#             data_one.at[0, 'Power'] = -value_['Power'].mean() # (because of the direction of thrust)
#             data_one.at[0, 'Torque'] = -value_['Torque'].mean() # (because of the direction of thrust)

#             # ignore the chord and twist data because the order is not sure
#             data_list.append(data_one)

#     # geometry data has been stored, now store the chord and twist data 
#     if chord is not None and twist is not None:
#         for data_one in data_list:
#             for i in range(22):
#                 data_one.at[0, f'chord_{i}'] = chord[i]
#                 data_one.at[0, f'twist_{i}'] = twist[i]
#             data_total = pd.concat([data_total, data_one], ignore_index=True)
# # save the data
# data_path = os.path.join('./data_for_train', 'dealed_data.xlsx')
# data_total.to_excel(data_path)

#### standard the data

In [25]:
# ==============================
# 加载数据
# ==============================
df = pd.read_excel('./data_for_train/dealed_data.xlsx')

# ==============================
# 定义输入输出列（仅使用运行工况作为输入）
# ==============================
input_columns = ['RPM', 'WIND', 'ANGLE']
output_columns = ['Fx', 'Fy', 'Fz', 'Torque']

X = df[input_columns]
Y = df[output_columns]

# ==============================
# 切分数据集
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# ==============================
# 标准化（输入/输出各用一个 Scaler）
# ==============================
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
Y_train_scaled = scaler_Y.fit_transform(y_train)
Y_test_scaled = scaler_Y.transform(y_test)

# ==============================
# 保存为 pickle
# ==============================
pd.DataFrame(X_train_scaled, columns=input_columns).to_pickle('./data_for_train/X_train_scaled')
pd.DataFrame(X_test_scaled, columns=input_columns).to_pickle('./data_for_train/X_test_scaled')
pd.DataFrame(Y_train_scaled, columns=output_columns).to_pickle('./data_for_train/y_train_scaled')
pd.DataFrame(Y_test_scaled, columns=output_columns).to_pickle('./data_for_train/y_test_scaled')

# ==============================
# 保存标准化器
# ==============================
joblib.dump(scaler_X, './data_for_train/scaler_X.pkl')
joblib.dump(scaler_Y, './data_for_train/scaler_Y.pkl')

print(f"训练集: {X_train_scaled.shape[0]} 条, 测试集: {X_test_scaled.shape[0]} 条")
print(f"输入维度: {X_train_scaled.shape[1]}, 输出维度: {Y_train_scaled.shape[1]}")

['./data_for_train/Torque_scaler.pkl']

## Hyperparameters

In [26]:
# ==============================
# 超参数配置
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PropellerPredictor(input_dim=3, output_dim=4).to(device)

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=200, min_lr=1e-6)

batch_size = 64
epochs = 5000
early_stop_patience = 500

print(f"设备: {device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")


## traing

### eval for tensorboard recording

In [27]:
# ==============================
# 评价函数
# ==============================
def evaluate_metrics(y_true, y_pred):
    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()
    mae = mean_absolute_error(y_true_np, y_pred_np)
    r2 = r2_score(y_true_np, y_pred_np)
    rel_error = np.abs((y_pred_np - y_true_np) / (np.abs(y_true_np) + 1e-8))
    return mae, r2, rel_error

### load data

In [28]:
# ==============================
# 加载数据，均为预处理数据(标准化)
# ==============================
X_train = torch.tensor(pd.read_pickle('./data_for_train/X_train_scaled').values, dtype=torch.float32)
X_test = torch.tensor(pd.read_pickle('./data_for_train/X_test_scaled').values, dtype=torch.float32)
y_train = torch.tensor(pd.read_pickle('./data_for_train/y_train_scaled').values, dtype=torch.float32)
y_test = torch.tensor(pd.read_pickle('./data_for_train/y_test_scaled').values, dtype=torch.float32)

# ==============================
# 分批次加载数据集
# ==============================
# dataloader setting
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

In [29]:
best_test_loss = float('inf')
no_improve_count = 0
output_names = ['Fx', 'Fy', 'Fz', 'Torque']

for epoch in range(epochs):
    # ---- 训练阶段 ----
    model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_X)
        loss = loss_fn(output, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * batch_X.size(0)

    avg_loss = total_loss / len(train_loader.dataset)

    # ---- 验证阶段 ----
    model.eval()
    with torch.no_grad():
        train_outputs = model(X_train.to(device))
        test_outputs = model(X_test.to(device))
        train_loss = loss_fn(train_outputs, y_train.to(device)).item()
        test_loss = loss_fn(test_outputs, y_test.to(device)).item()

        writer.add_scalar("Loss/Train", train_loss, epoch)
        writer.add_scalar("Loss/Test", test_loss, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]['lr'], epoch)

        train_mae, train_r2, train_rel = evaluate_metrics(y_train, train_outputs.cpu())
        test_mae, test_r2, test_rel = evaluate_metrics(y_test, test_outputs.cpu())
        writer.add_scalar("MAE/Train", train_mae, epoch)
        writer.add_scalar("MAE/Test", test_mae, epoch)
        writer.add_scalar("R2/Train", train_r2, epoch)
        writer.add_scalar("R2/Test", test_r2, epoch)

        for i, name in enumerate(output_names):
            writer.add_scalar(f'RelativeError/Train/{name}', np.mean(train_rel[:, i]), epoch)
            writer.add_scalar(f'RelativeError/Test/{name}', np.mean(test_rel[:, i]), epoch)

    # ---- 学习率调度 ----
    scheduler.step(test_loss)

    # ---- 早停机制 ----
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        no_improve_count = 0
        torch.save(model.state_dict(), "trained_models/best_model.pth")
    else:
        no_improve_count += 1

    if no_improve_count >= early_stop_patience:
        print(f"[早停] Epoch {epoch}, 测试集 loss 连续 {early_stop_patience} 轮未改善，停止训练")
        break

    if epoch % 500 == 0:
        print(f"[Epoch {epoch:>4d}] Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f} | "
              f"R²: {test_r2:.4f} | MAE: {test_mae:.6f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

print(f"\n训练完成，最佳测试 Loss: {best_test_loss:.6f}")
    

[Epoch 0] Loss: 0.220382
Test MAE: 0.347767
Test R²: -3.182084
[Epoch 500] Loss: 0.000886
Test MAE: 0.019508
Test R²: 0.984166
[Epoch 1000] Loss: 0.000651
Test MAE: 0.014892
Test R²: 0.990871
[Epoch 1500] Loss: 0.000609
Test MAE: 0.014857
Test R²: 0.991211
[Epoch 2000] Loss: 0.000640
Test MAE: 0.012473
Test R²: 0.993617
[Epoch 2500] Loss: 0.000766
Test MAE: 0.012767
Test R²: 0.992101
[Epoch 3000] Loss: 0.000663
Test MAE: 0.015702
Test R²: 0.989398
[Epoch 3500] Loss: 0.000686
Test MAE: 0.013925
Test R²: 0.991482
[Epoch 4000] Loss: 0.000665
Test MAE: 0.012092
Test R²: 0.993541
[Epoch 4500] Loss: 0.000566
Test MAE: 0.014280
Test R²: 0.990702
[Epoch 5000] Loss: 0.000608
Test MAE: 0.012314
Test R²: 0.992690
[Epoch 5500] Loss: 0.000666
Test MAE: 0.014847
Test R²: 0.990911
[Epoch 6000] Loss: 0.000625
Test MAE: 0.013237
Test R²: 0.991574
[Epoch 6500] Loss: 0.000718
Test MAE: 0.014244
Test R²: 0.991472
[Epoch 7000] Loss: 0.000681
Test MAE: 0.013424
Test R²: 0.991649
[Epoch 7500] Loss: 0.000598


In [30]:
## save the model to disk

In [31]:
os.makedirs("trained_models", exist_ok=True)
model.load_state_dict(torch.load("trained_models/best_model.pth"))
torch.save(model.state_dict(), "trained_models/propeller_predictor.pth")
writer.close()
print("已加载最佳模型并保存")

# prediction

## load the scaler

In [32]:
scaler_X = joblib.load('./data_for_train/scaler_X.pkl')
scaler_Y = joblib.load('./data_for_train/scaler_Y.pkl')

In [33]:
# ==============================
# 加载模型并预测
# ==============================
X_test_df = pd.read_pickle('./data_for_train/X_test_scaled')
y_test_df = pd.read_pickle('./data_for_train/y_test_scaled')

model.load_state_dict(torch.load('trained_models/propeller_predictor.pth'))
model.eval()

X_test_tensor = torch.tensor(X_test_df.values, dtype=torch.float32).to(device)
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).cpu().numpy()

# ==============================
# 反标准化 → 原始物理量
# ==============================
y_pred_original = scaler_Y.inverse_transform(y_pred_scaled)
y_true_original = scaler_Y.inverse_transform(y_test_df.values)

df_result = pd.DataFrame(y_pred_original, columns=['Fx_pred', 'Fy_pred', 'Fz_pred', 'Torque_pred'])
df_true = pd.DataFrame(y_true_original, columns=['Fx_true', 'Fy_true', 'Fz_true', 'Torque_true'])

# ==============================
# 计算相对误差
# ==============================
with np.errstate(divide='ignore', invalid='ignore'):
    relative_error = np.where(
        np.abs(y_true_original) > 1e-6,
        (y_true_original - y_pred_original) * 100 / y_true_original,
        0.0
    )
df_rel_err = pd.DataFrame(relative_error, columns=['Fx_err%', 'Fy_err%', 'Fz_err%', 'Torque_err%'])

df_all = pd.concat([df_true, df_result, df_rel_err], axis=1)
df_all.to_excel('./data_for_train/predict_vs_true.xlsx', index=False)

# ==============================
# 打印每个输出的 R² 和平均绝对误差
# ==============================
for i, name in enumerate(['Fx', 'Fy', 'Fz', 'Torque']):
    r2 = r2_score(y_true_original[:, i], y_pred_original[:, i])
    mae = mean_absolute_error(y_true_original[:, i], y_pred_original[:, i])
    print(f"{name:>6s}: R² = {r2:.4f}, MAE = {mae:.6f}, 平均相对误差 = {np.mean(np.abs(relative_error[:, i])):.2f}%")



In [34]:
print(df_all.max())

Fx_true                     7.772109
Fy_true                     0.083090
Fz_true                     0.560043
Torque_true                 0.156934
Fx_pred                     7.734492
Fy_pred                     0.079044
Fz_pred                     0.525560
Torque_pred                 0.147942
Fx_relative_error           6.107889
Fy_relative_error        2522.231518
Fz_relative_error           6.157135
Torque_relative_error       6.100975
dtype: float64
